In [2]:
!pip install keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.6 MB/s eta 0:00:00


In [4]:
# import keras

In [2]:
# import os
# import numpy as np
# from PIL import Image
# from keras.preprocessing.image import img_to_array, load_img

# def load_custom_dataset(dataset_path, image_size):
#     """
#     Load and preprocess a custom dataset.

#     Args:
#     - dataset_path: Path to the directory containing the dataset.
#     - image_size: Tuple specifying the target image size (e.g., (64, 64)).

#     Returns:
#     - A numpy array containing the preprocessed images.
#     """
#     # Initialize an empty list to store preprocessed images
#     images = []

#     # Iterate over the files in the dataset directory
#     for filename in os.listdir(dataset_path):
#         # Construct the full path to the image file
#         filepath = os.path.join(dataset_path, filename)

#         # Load the image and resize it to the target size
#         image = load_img(filepath, target_size=image_size)

#         # Convert the image to a numpy array and normalize the pixel values
#         image = img_to_array(image) / 255.0

#         # Append the preprocessed image to the list
#         images.append(image)

#     # Convert the list of images to a numpy array
#     images = np.array(images)

#     return images

# # Example usage:
# dataset_path = './archive/resized_data/images/test'
# image_size = (640, 640,3)
# custom_dataset = load_custom_dataset(dataset_path, image_size)
# print("Custom dataset shape:", custom_dataset.shape)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
# Generator model nural networks
class Generator(nn.Module):
    def __init__(self, latent_dim, img_shape):
        super(Generator, self).__init__()
        self.img_shape = img_shape
        input_size = img_shape[0] * img_shape[1] * img_shape[2]
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128), # linear transform
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, input_size), 
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), *self.img_shape)
        return img
# Discriminator model
class Discriminator(nn.Module):
    def __init__(self, img_shape):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(torch.prod(torch.tensor(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

# Hyperparameters
latent_dim = 10000
img_shape = (3, 640, 640)
lr = 0.0002
batch_size = 8
epochs = 11
# self.input_size = img_shape[0] * img_shape[1] * img_shape[2]
# Initialize models
generator = Generator(latent_dim, img_shape)
discriminator = Discriminator(img_shape)

# Loss function
adversarial_loss = nn.BCELoss()

# Optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

# DataLoader
transform = transforms.Compose([transforms.Resize((640, 640)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.ImageFolder(root='./archive/resized_data/images/test', transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop
for epoch in range(epochs):
    for i, (imgs, _) in enumerate(dataloader):
        # Adversarial ground truths
        valid = torch.ones(imgs.size(0), 1)
        fake = torch.zeros(imgs.size(0), 1)
        # Configure input
        real_imgs = imgs.view(imgs.size(0), -1)
        z = torch.randn(imgs.size(0), latent_dim)
        #  Train Discriminator
        optimizer_D.zero_grad()
        # Loss on real images
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        # Loss on fake images
        fake_imgs = generator(z)
        fake_loss = adversarial_loss(discriminator(fake_imgs.detach()), fake)
        # Total discriminator loss
        d_loss = (real_loss + fake_loss) / 2

        d_loss.backward()
        optimizer_D.step()
        #  Train Generator
        optimizer_G.zero_grad()
        # Generate fake images
        gen_imgs = generator(z)
        # Loss measures generator's ability to fool the discriminator
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)

        g_loss.backward()
        optimizer_G.step()

        if i % 100 == 0:
            print(
                "[Epoch %d/%d] [Batch %d/%d] [D loss: %f] [G loss: %f]"
                % (epoch, epochs, i, len(dataloader), d_loss.item(), g_loss.item())
            )


# Generate and save sample images
z = torch.randn(50, latent_dim)
gen_imgs = generator(z)
# Reshape generated images 
gen_imgs = gen_imgs.view(50, 3, 640, 640)
save_dir = "./ganimage"
os.makedirs(save_dir, exist_ok=True)
# gein_imgs = gen_imgs.cuda() if torch.cuda.is_available() else gen_imgs

for i in range(50):
    #plt.subplot(100, 100, i + 1)
    plt.imshow(gen_imgs[i].detach().cpu().numpy().transpose((1, 2, 0)))
    plt.axis('off')
    plt.savefig(os.path.join(save_dir, f'gan_images_{i}.jpg'))
    plt.clf()

plt.imshow(gen_imgs[-1].detach().cpu().numpy().transpose((1, 2, 0)))
plt.axis('off')
plt.show()


[Epoch 0/11] [Batch 0/1] [D loss: 0.694744] [G loss: 24.958511]
[Epoch 1/11] [Batch 0/1] [D loss: 0.000000] [G loss: 33.209747]
[Epoch 2/11] [Batch 0/1] [D loss: 0.000000] [G loss: 36.875568]
[Epoch 3/11] [Batch 0/1] [D loss: 0.000000] [G loss: 36.938480]
[Epoch 4/11] [Batch 0/1] [D loss: 0.000000] [G loss: 33.934246]
[Epoch 5/11] [Batch 0/1] [D loss: 0.000000] [G loss: 27.932104]
[Epoch 6/11] [Batch 0/1] [D loss: 0.019740] [G loss: 25.355562]
[Epoch 7/11] [Batch 0/1] [D loss: 0.065179] [G loss: 25.545174]
[Epoch 8/11] [Batch 0/1] [D loss: 1.283593] [G loss: 35.289875]
[Epoch 9/11] [Batch 0/1] [D loss: 0.072271] [G loss: 45.621124]
[Epoch 10/11] [Batch 0/1] [D loss: 0.000000] [G loss: 54.541897]


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).
Clipping i

<Figure size 640x480 with 1 Axes>

In [ ]:
import os

def generate_and_save_images(generator, epoch, save_dir='generated_images'):
    # Create the directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    noise = np.random.normal(0, 1, (25, 100))  # Generate random noise
    gen_imgs = generator.predict(noise)  # Generate images

    # Rescale images 0 - 255
    gen_imgs = 127.5 * gen_imgs + 127.5
    gen_imgs = gen_imgs.astype(np.uint8)

    # Create a grid of 5x5 images
    fig, axs = plt.subplots(640, 640, figsize=(640, 640))
    idx = 0
    for i in range(5):
        for j in range(5):
            img = gen_imgs[idx]
            axs[i, j].imshow(img)
            axs[i, j].axis('off')
            idx += 1
    plt.tight_layout()
    
    # Save the grid as an image
    plt.savefig(os.path.join(save_dir, f'generated_images_{epoch}.jpg'))
    plt.close()

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# Generator model
class Generator(nn.Module):
    def __init__(self, latent_dim, img_shape):
        super(Generator, self).__init__()
        self.img_shape = img_shape
        # Calculate the total number of input features
        input_size = img_shape[0] * img_shape[1] * img_shape[2]
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, input_size),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), *self.img_shape)
        return img

# Discriminator model
class Discriminator(nn.Module):
    def __init__(self, img_shape):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(torch.prod(torch.tensor(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

# Hyperparameters
latent_dim = 100
img_shape = (3, 640, 640)
lr = 0.0002
batch_size = 8
epochs = 10

# Initialize models
generator = Generator(latent_dim, img_shape)
discriminator = Discriminator(img_shape)

# Loss function
adversarial_loss = nn.BCELoss()

# Optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

# DataLoader
transform = transforms.Compose([transforms.Resize((640, 640)), transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.ImageFolder(root='./archive/resized_data/images/test', transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop
for epoch in range(epochs):
    for i, (imgs, _) in enumerate(dataloader):
        # Adversarial ground truths
        valid = torch.ones(imgs.size(0), 1)
        fake = torch.zeros(imgs.size(0), 1)

        # Configure input
        real_imgs = imgs.view(imgs.size(0), -1)
        z = torch.randn(imgs.size(0), latent_dim)

        # Train Discriminator
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_imgs = generator(z)
        fake_loss = adversarial_loss(discriminator(fake_imgs.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # Train Generator
        optimizer_G.zero_grad()
        gen_imgs = generator(z)
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

        if i % 100 == 0:
            print(
                "[Epoch %d/%d] [Batch %d/%d] [D loss: %f] [G loss: %f]"
                % (epoch, epochs, i, len(dataloader), d_loss.item(), g_loss.item())
            )

# Generate and save sample images
z = torch.randn(25, latent_dim)
gen_imgs = generator(z)

# Specify the directory where you want to save the images
save_dir = "./ganimage"

# Create the directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Plot and save the generated images
fig, axs = plt.subplots(5, 5, figsize=(10, 10))

for i in range(1):
    row = i // 1
    col = i % 1
    img = gen_imgs[i].detach().cpu().numpy().transpose((1, 2, 0))
    axs[row, col].imshow(img)
    axs[row, col].axis('off')

# Save the generated images in the specified directory
plt.savefig(os.path.join(save_dir, f'gan_generated_images_{epoch}.jpg'))
plt.show()


[Epoch 0/10] [Batch 0/1] [D loss: 0.692607] [G loss: 29.654039]
[Epoch 1/10] [Batch 0/1] [D loss: 0.000000] [G loss: 43.211010]
[Epoch 2/10] [Batch 0/1] [D loss: 0.000000] [G loss: 56.090668]
[Epoch 3/10] [Batch 0/1] [D loss: 0.000000] [G loss: 66.395912]
[Epoch 4/10] [Batch 0/1] [D loss: 0.000000] [G loss: 76.290718]
[Epoch 5/10] [Batch 0/1] [D loss: 0.000000] [G loss: 85.538086]
[Epoch 6/10] [Batch 0/1] [D loss: 0.000000] [G loss: 93.850555]
[Epoch 7/10] [Batch 0/1] [D loss: 0.000000] [G loss: 93.919861]
[Epoch 8/10] [Batch 0/1] [D loss: 0.000000] [G loss: 91.035698]
[Epoch 9/10] [Batch 0/1] [D loss: 0.000000] [G loss: 89.969826]


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers).


<Figure size 1000x1000 with 25 Axes>

In [3]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# import matplotlib.pyplot as plt
# import os

# # Generator model
# class Generator(nn.Module):
#     def __init__(self, latent_dim, img_shape):
#         super(Generator, self).__init__()
#         self.img_shape = img_shape
#         # Calculate the total number of input features
#         input_size = img_shape[0] * img_shape[1] * img_shape[2]
#         self.model = nn.Sequential(
#             nn.Linear(latent_dim, 128),
#             nn.LeakyReLU(0.2, inplace=True),
#             nn.Linear(128, 256),
#             nn.BatchNorm1d(256),
#             nn.LeakyReLU(0.2, inplace=True),
#             nn.Linear(256, 512),
#             nn.BatchNorm1d(512),
#             nn.LeakyReLU(0.2, inplace=True),
#             nn.Linear(512, input_size),
#             nn.Tanh()
#         )

#     def forward(self, z):
#         img = self.model(z)
#         img = img.view(img.size(0), *self.img_shape)
#         return img

# # Discriminator model
# class Discriminator(nn.Module):
#     def __init__(self, img_shape):
#         super(Discriminator, self).__init__()
#         self.model = nn.Sequential(
#             nn.Linear(torch.prod(torch.tensor(img_shape)), 512),
#             nn.LeakyReLU(0.2, inplace=True),
#             nn.Linear(512, 256),
#             nn.LeakyReLU(0.2, inplace=True),
#             nn.Linear(256, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, img):
#         img_flat = img.view(img.size(0), -1)
#         validity = self.model(img_flat)
#         return validity

# # Hyperparameters
# latent_dim = 100
# img_shape = (3, 640, 640)
# lr = 0.0002
# batch_size = 8
# epochs = 10

# # Initialize models
# generator = Generator(latent_dim, img_shape)
# discriminator = Discriminator(img_shape)

# # Loss function
# adversarial_loss = nn.BCELoss()

# # Optimizers
# optimizer_G = optim.Adam(generator.parameters(), lr=lr)
# optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

# # DataLoader
# transform = transforms.Compose([transforms.Resize((640, 640),transforms.ToTensor())])
# dataset = datasets.ImageFolder(root='./archive/resized_data/images/test', transform=transform)
# dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# # Training loop
# for epoch in range(epochs):
#     for i, (imgs, _) in enumerate(dataloader):
#         # Adversarial ground truths
#         valid = torch.ones(imgs.size(0), 1)
#         fake = torch.zeros(imgs.size(0), 1)

#         # Configure input
#         real_imgs = imgs.view(imgs.size(0), -1)
#         z = torch.randn(imgs.size(0), latent_dim)

#         # Train Discriminator
#         optimizer_D.zero_grad()
#         real_loss = adversarial_loss(discriminator(real_imgs), valid)
#         fake_imgs = generator(z)
#         fake_loss = adversarial_loss(discriminator(fake_imgs.detach()), fake)
#         d_loss = (real_loss + fake_loss) / 2
#         d_loss.backward()
#         optimizer_D.step()

#         # Train Generator
#         optimizer_G.zero_grad()
#         gen_imgs = generator(z)
#         g_loss = adversarial_loss(discriminator(gen_imgs), valid)
#         g_loss.backward()
#         optimizer_G.step()

#         if i % 100 == 0:
#             print(
#                 "[Epoch %d/%d] [Batch %d/%d] [D loss: %f] [G loss: %f]"
#                 % (epoch, epochs, i, len(dataloader), d_loss.item(), g_loss.item())
#             )

# # Generate and save sample images
# z = torch.randn(25, latent_dim)
# gen_imgs = generator(z)

# # Specify the directory where you want to save the images
# save_dir = "./ganimage"

# # Create the directory if it doesn't exist
# os.makedirs(save_dir, exist_ok=True)

# # Plot and save the generated images
# fig, axs = plt.subplots(5, 5, figsize=(10, 10))

# for i in range(25):
#     row = i // 5
#     col = i % 5
#     img = gen_imgs[i].detach().cpu().numpy().transpose((1, 2, 0))
#     axs[row, col].imshow(img)
#     axs[row, col].axis('off')

# # Save the generated images in the specified directory
# plt.savefig(os.path.join(save_dir, f'gan_generated_images_{epoch}.jpg'))
# plt.show()


In [ ]:
import os

def generate_and_save_images(generator, epoch, save_dir='generated_images'):
    # Create the directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    noise = np.random.normal(0, 1, (25, 100))  # Generate random noise
    gen_imgs = generator.predict(noise)  # Generate images

    # Rescale images 0 - 255
    gen_imgs = 127.5 * gen_imgs + 127.5
    gen_imgs = gen_imgs.astype(np.uint8)

    # Create a grid of 5x5 images
    fig, axs = plt.subplots(640, 640, figsize=(640, 640))
    idx = 0
    for i in range(5):
        for j in range(5):
            img = gen_imgs[idx]
            axs[i, j].imshow(img)
            axs[i, j].axis('off')
            idx += 1
    plt.tight_layout()
    
    # Save the grid as an image
    plt.savefig(os.path.join(save_dir, f'generated_images_{epoch}.jpg'))
    plt.close()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential, Model
from keras.layers import Dense, Reshape, Flatten, Conv2D, Conv2DTranspose, LeakyReLU, Input
from keras.optimizers import Adam
from PIL import Image
import os

# Generator model
def build_generator():
    model = Sequential()
    model.add(Dense(16 * 16 * 256, input_shape=(100,)))
    model.add(Reshape((16, 16, 256)))
    model.add(Conv2DTranspose(128, kernel_size=3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2DTranspose(64, kernel_size=3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2DTranspose(3, kernel_size=3, strides=2, padding='same', activation='tanh'))
    return model

# Function to generate and save images

def generate_and_save_images(generator, epoch, save_dir='./ganimage'):
    # Create the directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    noise = np.random.normal(0, 1, (25, 100))  # Generate random noise
    gen_imgs = generator.predict(noise)  # Generate images

    # Rescale images 0 - 255
    gen_imgs = 127.5 * gen_imgs + 127.5
    gen_imgs = gen_imgs.astype(np.uint8)

    # Create a grid of 5x5 images
    fig, axs = plt.subplots(5, 5, figsize=(10, 10))
    idx = 0
    for i in range(5):
        for j in range(5):
            img = gen_imgs[idx]
            axs[i, j].imshow(img)
            axs[i, j].axis('off')
            idx += 1
    plt.tight_layout()
    
    # Save the grid as an image
    plt.savefig(os.path.join(save_dir, f'generated_images_{epoch}.jpg'))
    plt.close()

# Main function
def main():
    # Generator model
    generator = build_generator()

    # Compile the generator
    generator.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))

    # Training loop
    epochs = 10000
    batch_size = 8
    sample_interval = 1000

    for epoch in range(epochs):
        # Generate a batch of fake images
        noise = np.random.normal(0, 1, (batch_size, 100))
        gen_imgs = generator.predict(noise)

        # Train the generator (no actual data needed for this simplified example)
        g_loss = generator.train_on_batch(noise, np.ones((batch_size, 1)))

        # Print progress
        print ("%d [G loss: %f]" % (epoch, g_loss))

        # Save generated images at sample intervals
        if epoch % sample_interval == 0:
            generate_and_save_images(generator, epoch)

if __name__ == "__main__":
    main()
